**Objective**

Clean the combined master_metadata.csv and produce a standardized dataset for EDA.

Input: data/processed/master_metadata.csv

Output:
data/processed/cleaned_metadata.csv
reports/data_cleaning_log.csv

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

In [ ]:
PROJECT_DIR = Path("/workspaces/DBA-ai-trust-score-framework-Interim")

PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
REPORT_DIR = PROJECT_DIR / "reports"

REPORT_DIR.mkdir(exist_ok=True)

In [ ]:
metadata = pd.read_csv(
    PROCESSED_DIR / "master_metadata.csv"
)

metadata.head()

In [ ]:
print("Shape :", metadata.shape)

print("\nColumns:")
print(metadata.columns.tolist())

print("\nData Types:")
print(metadata.dtypes)

In [ ]:
missing = metadata.isnull().sum()

missing = missing[
    missing > 0
].sort_values(
    ascending=False
)

missing

________________Duplicate Records_________________

In [ ]:
duplicates = metadata.duplicated().sum()

print("Duplicate records:", duplicates)

_________________Remove Duplicates____________________

In [ ]:
metadata = metadata.drop_duplicates()

print(metadata.shape)

__________________Standardize column names_________________

In [ ]:
metadata.columns = (
    metadata.columns
    .str.lower()
    .str.strip()
    .str.replace(" ", "_")
)

metadata.columns

______________________Removing Empty cells____________________

In [ ]:
metadata = metadata.dropna(
    axis=1,
    how="all"
)

metadata.shape

________________________Fill Missing Cells________________________

In [ ]:
import pandas as pd

for col in metadata.columns:

    # Numeric columns
    if pd.api.types.is_numeric_dtype(metadata[col]):

        metadata[col] = metadata[col].fillna(
            metadata[col].median()
        )

    # Everything else
    else:

        metadata[col] = metadata[col].fillna("Unknown")

__________________________Data Cleaning Log________________________

In [ ]:
cleaning_log = pd.DataFrame({

    "Issue":[

        "Missing Values",
        "Duplicate Records",
        "Columns Removed"

    ],

    "Count":[

        missing.sum(),
        duplicates,
        0

    ],

    "Treatment":[

        "Median/Unknown Imputation",
        "Removed",
        "None"

    ]

})

cleaning_log

__________________________Save Outputs_______________________

In [ ]:
metadata.to_csv(
    PROCESSED_DIR / "cleaned_metadata.csv",
    index=False
)

cleaning_log.to_csv(
    REPORT_DIR / "data_cleaning_log.csv",
    index=False
)

print("✅ cleaned_metadata.csv saved")
print("✅ data_cleaning_log.csv saved")

___________________________Final Validation________________________

In [ ]:
print("Final Shape:", metadata.shape)

print("\nMissing Values Remaining")

print(
    metadata.isnull().sum().sum()
)

print("\nDuplicate Records")

print(
    metadata.duplicated().sum()
)